In [3]:
import os
from typing import List, Tuple, Any

from pydantic import ConfigDict
from pinecone import Pinecone

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_pinecone import PineconeVectorStore
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# -----------------------------
# 세션별 chat history 저장
# -----------------------------
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# -----------------------------
# MultiPineconeRetriever 정의
# -----------------------------
class MultiPineconeRetriever(BaseRetriever):
    vectorstores: List[Any]
    k_each: int = 5
    final_k: int = 5
    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        all_docs_with_scores: List[Tuple[Document, float]] = []

        for vs in self.vectorstores:
            # docs_with_scores = vs.similarity_search_with_score(query, k=self.k_each)
            # docs_with_scores = vs.similarity_search_with_score([query], k=self.k_each)
            # all_docs_with_scores.extend(docs_with_scores)
            docs_with_scores = vs.similarity_search_with_score(query, k=self.k_each)  # ✅ 수정됨
            all_docs_with_scores.extend(docs_with_scores)

        all_docs_with_scores = sorted(
            all_docs_with_scores,
            key=lambda x: x[1],
            reverse=True,
        )

        top_docs = []
        for rank, (doc, score) in enumerate(all_docs_with_scores[: self.final_k], start=1):
            doc.metadata["retrieval_score"] = score
            doc.metadata["retrieval_rank"] = rank
            top_docs.append(doc)
        return top_docs


# -----------------------------
# 문서 포맷 함수
# -----------------------------
def format_docs(docs: List[Document]) -> str:
    formatted_docs = []
    for doc in docs:
        regulation = doc.metadata.get("regulation_name", "")
        article = doc.metadata.get("article", "")
        text = f"""
규정명 : {regulation}
조문: {article} 
본문:
{doc.page_content}
"""
        formatted_docs.append(text)
    return "\n\n".join(formatted_docs)


# -----------------------------
# extract_metadata
# -----------------------------
def extract_metadata(data):
    docs = data["docs"]
    question = data["input"]

    if not docs:
        return {
            "context": "",
            "regulation": "관련 규정 또는 기술기준 확인 불가",
            "article": "",
            "input": question,
        }

    top_doc = docs[0]

    return {
        "context": format_docs(docs),
        "regulation": top_doc.metadata.get("regulation_name", "관련 규정 또는 기술기준 확인 불가"),
        "article": top_doc.metadata.get("article", ""),
        "input": question,
    }


# -----------------------------
# LLM 호출
# -----------------------------
def get_llm(model='upstage'):
    if model == 'upstage':
        llm = ChatUpstage()
    else:
        raise ValueError(f"Invalid model: {model}")
    return llm


# -----------------------------
# Pinecone Retriever 설정
# -----------------------------
def get_retrieved_docs():
    embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

    UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    GROUNDLINE_INDEX = os.getenv("GROUNDLINE_INDEX")
    BROADCOM_INDEX = os.getenv("BROADCOM_INDEX")
    PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

    if not UPSTAGE_API_KEY:
        raise ValueError("UPSTAGE_API_KEY가 없습니다.")
    if not PINECONE_API_KEY:
        raise ValueError("PINECONE_API_KEY가 없습니다.")
    if not GROUNDLINE_INDEX or not BROADCOM_INDEX:
        raise ValueError("GROUNDLINE_INDEX / BROADCOM_INDEX가 필요합니다.")

    pc = Pinecone(api_key=PINECONE_API_KEY)
    groundline_index = pc.Index(GROUNDLINE_INDEX)
    broadcom_index = pc.Index(BROADCOM_INDEX)

    groundline_db = PineconeVectorStore(
        index=groundline_index,
        embedding=embeddings,
        namespace=PINECONE_NAMESPACE,
    )
    broadcom_db = PineconeVectorStore(
        index=broadcom_index,
        embedding=embeddings,
        namespace=PINECONE_NAMESPACE,
    )

    multi_retriever = MultiPineconeRetriever(
        vectorstores=[groundline_db, broadcom_db],
        k_each=5,
        final_k=5,
    )

    return multi_retriever


# -----------------------------
# AI 메시지 생성 함수 (session_id 포함)
# -----------------------------
def get_ai_message(user_message, session_id: str = "user_1"):
    llm = get_llm()
    retrieved_docs = get_retrieved_docs()

    user_message_clean = user_message.strip()

    # QA 프롬프트
    qa_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
너는 통신설비 기술 질의응답 전문가다.
반드시 제공된 context만 근거로 답변하라.
context에 없으면 추측하지 말고 "관련 규정 또는 기술기준 확인 불가, 제공된 문서에서 확인되지 않습니다."라고 답하라.
답변은 한국어로, 핵심부터 간결하게 작성하라.

반드시 아래 형식으로만 답하라.

{regulation} {article}, 답변내용

예시:
접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준 제8조, 통신설비의 접지저항은 10Ω 이하로 해야 합니다.

<context>
{context}
</context>
""".strip(),
            ),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ]
    )

    # RAG 체인 구성
    rag_chain = (
        {
            "docs": retrieved_docs,
            "input": RunnablePassthrough(),
        }
        | RunnableLambda(extract_metadata)
        | qa_prompt
        | llm
    )

    # RunnableWithMessageHistory로 래핑
    rag_chain_with_history = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history"
    )

    # invoke 시 session_id 반드시 전달
    # ai_message = rag_chain_with_history.invoke(
    #     {"input": user_message},
    #     {"configurable": {"session_id": session_id}}
    # )

    ai_message = rag_chain_with_history.invoke(
        {"input": user_message_clean},  # ✅ 수정됨
        {"configurable": {"session_id": session_id}}
    )
    
    return ai_message.content


# -----------------------------
# 테스트
# -----------------------------
# if __name__ == "__main__":
#     session_id = "test_user_001"
#     query = "통신설비의 접지저항은 얼마입니까?"
#     response = get_ai_message(query, session_id=session_id)
#     print("AI Response:\n", response)

In [2]:
# -----------------------------
# 테스트
# -----------------------------
if __name__ == "__main__":
    session_id = "test_user_001"
    query = "통신설비의 접지저항은 얼마입니까?"
    response = get_ai_message(query, session_id=session_id)
    print("AI Response:\n", response)

BadRequestError: Error code: 400 - {'error': {'message': "'$.input' is invalid. Please check the API reference: https://developers.upstage.ai/docs/apis/embeddings", 'type': 'invalid_request_error', 'param': 'input', 'code': None}}